# Module 3 — Clinical Trial Risk & Scenario Modelling (ClinicalTrials.gov)

**Scenario**: A trial sponsor wants (1) a risk score per trial in a therapeutic
area — flagging trials at risk of termination, withdrawal, or under-reporting —
and (2) a Monte Carlo "what-if" simulation of how long it will take a currently
recruiting trial to complete enrollment.

**This notebook**:
1. Pulls real trial records for several related conditions from ClinicalTrials.gov
2. Trains a termination-risk classifier (cross-validated) and persists it to `models/`
3. Builds a per-trial risk score (termination/withdrawal risk, reporting risk, enrollment scale)
4. Derives a benchmark monthly-enrollment rate from completed trials in the same therapeutic area
5. Runs a Monte Carlo simulation for a currently-recruiting trial's enrollment completion
6. Writes a stakeholder summary to `reports/clinicaltrials_summary.md`

**Caveat**: ClinicalTrials.gov is a *registry*, not a performance database — "enrollment
count" is the latest reported figure (often the target, not actuals-to-date), and many
trials have missing or `"YYYY"`/`"YYYY-MM"`-only dates. The benchmark rate below is a
rough sponsor-level planning estimate, not a guarantee.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data_connectors.clinicaltrials import fetch_studies
from src.risk_framework.risk_scoring import RiskScoringEngine
from src.risk_framework.scenario_simulation import simulate_enrollment_completion, summarise_simulation

THERAPEUTIC_AREA = "Diabetes"
CONDITIONS = [
    "Diabetes Mellitus, Type 2",
    "Diabetes Mellitus, Type 1",
    "Diabetic Nephropathy",
    "Diabetic Retinopathy",
    "Gestational Diabetes",
]

## 1. Pull trial records

A single condition's worth of trials gives too few terminated/withdrawn cases to
train a meaningful classifier, so we pull several closely related conditions and
deduplicate by NCT ID. Each pull is limited to a single page (100 studies) —
ClinicalTrials.gov pagination tokens are time-limited, so caching across pages
is unreliable for a notebook that may be re-run later.

In [2]:
frames = [fetch_studies(c, max_records=100) for c in CONDITIONS]
trials = pd.concat(frames, ignore_index=True).drop_duplicates(subset="nct_id").reset_index(drop=True)

for col in ["start_date", "primary_completion_date", "completion_date"]:
    trials[col + "_dt"] = pd.to_datetime(trials[col], errors="coerce")

print(f"Fetched {len(trials)} unique trials across {len(CONDITIONS)} related '{THERAPEUTIC_AREA}' conditions")
trials["overall_status"].value_counts()

Fetched 478 unique trials across 5 related 'Diabetes' conditions


overall_status
COMPLETED                  261
UNKNOWN                     81
RECRUITING                  52
TERMINATED                  28
NOT_YET_RECRUITING          23
WITHDRAWN                   14
ACTIVE_NOT_RECRUITING       13
ENROLLING_BY_INVITATION      5
SUSPENDED                    1
Name: count, dtype: int64

## 2. Trial termination-risk classifier

Train a classifier to predict P(trial is terminated, withdrawn, or suspended)
from characteristics known while a trial is active: target enrollment, phase,
study type, sponsor class, and FDA-regulation flags. Trials with status
`NOT_YET_RECRUITING` are excluded from training — they haven't had the chance
to terminate yet.

Only ~9-10% of eligible trials end up terminated/withdrawn/suspended, so a
single train/test split would give an unstable estimate with this few
positives — we use 5-fold stratified cross-validation instead. A
`RandomForestClassifier` is used (a linear model with `class_weight="balanced"`
was also tried and badly over-predicted terminations — ROC-AUC ~0.59 vs. ~0.83
for the random forest).

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

trials["enrollment_count"] = pd.to_numeric(trials["enrollment_count"], errors="coerce")
trials["target_terminated"] = trials["overall_status"].isin(
    ["TERMINATED", "WITHDRAWN", "SUSPENDED"]
).astype(int)
trials["phase_primary"] = trials["phases"].fillna("").replace("", "NONE").str.split(",").str[0]
trials["is_fda_regulated_drug"] = trials["is_fda_regulated_drug"].fillna(False).astype(str)
trials["is_fda_regulated_device"] = trials["is_fda_regulated_device"].fillna(False).astype(str)

TERM_NUM_FEATURES = ["enrollment_count"]
TERM_CAT_FEATURES = ["phase_primary", "study_type", "lead_sponsor_class",
                      "is_fda_regulated_drug", "is_fda_regulated_device"]

train_eligible = trials[trials["overall_status"] != "NOT_YET_RECRUITING"].copy()
print(f"Training on {len(train_eligible)} trials with a settled status "
      f"({int(train_eligible['target_terminated'].sum())} terminated/withdrawn/suspended, "
      f"{train_eligible['target_terminated'].mean():.1%})")

term_X = train_eligible[TERM_NUM_FEATURES + TERM_CAT_FEATURES]
term_y = train_eligible["target_terminated"]

termination_pipeline = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), TERM_NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), TERM_CAT_FEATURES),
    ])),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=5, class_weight="balanced", random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_preds = cross_val_predict(termination_pipeline, term_X, term_y, cv=cv)
cv_probs = cross_val_predict(termination_pipeline, term_X, term_y, cv=cv, method="predict_proba")[:, 1]

print(classification_report(term_y, cv_preds))
term_cv_roc_auc = roc_auc_score(term_y, cv_probs)
print(f"ROC-AUC (5-fold cross-validated): {term_cv_roc_auc:.4f}")

# Refit on the full eligible set; score every trial (including NOT_YET_RECRUITING) for the watchlist
termination_pipeline.fit(term_X, term_y)
trials["termination_risk_model"] = termination_pipeline.predict_proba(
    trials[TERM_NUM_FEATURES + TERM_CAT_FEATURES]
)[:, 1]

C:\Users\allif\AppData\Local\Temp\ipykernel_24336\3082820372.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  trials["is_fda_regulated_drug"] = trials["is_fda_regulated_drug"].fillna(False).astype(str)
C:\Users\allif\AppData\Local\Temp\ipykernel_24336\3082820372.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  trials["is_fda_regulated_device"] = trials["is_fda_regulated_device"].fillna(False).astype(str)


Training on 455 trials with a settled status (43 terminated/withdrawn/suspended, 9.5%)


              precision    recall  f1-score   support

           0       0.95      0.94      0.95       412
           1       0.49      0.51      0.50        43

    accuracy                           0.90       455
   macro avg       0.72      0.73      0.72       455
weighted avg       0.91      0.90      0.90       455

ROC-AUC (5-fold cross-validated): 0.8373


In [4]:
import json
import joblib
from datetime import datetime, timezone
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "clinical_trial_termination_classifier.joblib"
joblib.dump(termination_pipeline, model_path)

metadata = {
    "model": "RandomForestClassifier (n_estimators=200, max_depth=5, class_weight=balanced)",
    "purpose": "Predict P(trial terminated/withdrawn/suspended) for per-trial risk scoring",
    "therapeutic_area": THERAPEUTIC_AREA,
    "conditions_combined": CONDITIONS,
    "numeric_features": TERM_NUM_FEATURES,
    "categorical_features": TERM_CAT_FEATURES,
    "training_rows": len(train_eligible),
    "class_balance": {
        "terminated_withdrawn_suspended": int(term_y.sum()),
        "other": int((~term_y.astype(bool)).sum()),
    },
    "cv_method": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "cv_roc_auc": float(term_cv_roc_auc),
    "cv_classification_report": classification_report(term_y, cv_preds, output_dict=True),
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "caveats": [
        f"Only {int(term_y.sum())} of {len(train_eligible)} eligible trials "
        f"({term_y.mean():.1%}) are terminated/withdrawn/suspended — metrics are 5-fold "
        "cross-validated out-of-fold predictions; the minority-class precision/recall "
        "estimates carry wide uncertainty at this sample size.",
        "A LogisticRegression with class_weight='balanced' was also tried on this data and "
        "scored ROC-AUC ~0.59 (near random) while over-predicting terminations — "
        "RandomForestClassifier was chosen instead.",
        "'enrollment_count' on ClinicalTrials.gov is typically the target, not actual "
        "enrollment-to-date.",
    ],
}
with open(models_dir / "clinical_trial_termination_classifier.meta.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model to {model_path.resolve()}")
print(f"Saved metadata to {(models_dir / 'clinical_trial_termination_classifier.meta.json').resolve()}")

Saved model to C:\Users\allif\Downloads\Risk Modelling\models\clinical_trial_termination_classifier.joblib
Saved metadata to C:\Users\allif\Downloads\Risk Modelling\models\clinical_trial_termination_classifier.meta.json


## 3. Per-trial risk scoring

Three normalised (0-1) factors:
- **termination_risk_model**: predicted P(terminated/withdrawn/suspended) from the
  classifier above — a smoother signal than the raw binary outcome, and one that
  also produces a meaningful score for trials that haven't reached a settled status
- **reporting_risk**: 1 if the trial's status is UNKNOWN (sponsor stopped updating
  the registry — a compliance/oversight signal in its own right)
- **scale_risk**: enrollment size relative to the largest trial in this set (larger
  trials carry more operational/recruitment risk)

In [5]:
trials["reporting_risk"] = (trials["overall_status"] == "UNKNOWN").astype(float)
trials["scale_risk"] = (trials["enrollment_count"] / trials["enrollment_count"].max()).clip(upper=1.0)

engine = RiskScoringEngine(weights={"termination_risk_model": 0.5, "reporting_risk": 0.3, "scale_risk": 0.2})
scored = engine.score_dataframe(trials).sort_values("risk_score", ascending=False)
watchlist = scored[["nct_id", "brief_title", "overall_status", "why_stopped", "enrollment_count",
                     "termination_risk_model", "risk_score", "risk_band"]]
watchlist.head(10)

,nct_id,brief_title,overall_status,why_stopped,enrollment_count,termination_risk_model,risk_score,risk_band
54,NCT01565096,Effect of Adding Vildagliptin on Beta Cell Fun...,UNKNOWN,None,44.0,0.566675,0.5833,High
56,NCT02477761,Sustained Effects of a Non-glucidic Nutrient P...,UNKNOWN,None,7.0,0.529151,0.5646,High
347,NCT00673296,Intravitreal Injection of Bevacizumab and Gas ...,UNKNOWN,None,6.0,0.524253,0.5621,High
190,NCT01253928,"Pioglitazone, Body Composition,Insulin Sensiti...",UNKNOWN,None,16.0,0.425425,0.5127,High
292,NCT03859245,Photobiomodulation & Ketogenic Diet for Treatm...,UNKNOWN,None,30.0,0.381592,0.4908,Medium
349,NCT02353923,OcuStem Nutritional Supplement in Diabetic Pat...,UNKNOWN,None,30.0,0.381592,0.4908,Medium
472,NCT01855763,Metformin Treatment in Gestational Diabetes an...,UNKNOWN,None,300.0,0.372913,0.4865,Medium
294,NCT06572553,Evaluation of Therapeutic Efficacy of Faricima...,UNKNOWN,None,30.0,0.362384,0.4812,Medium
148,NCT01285245,Effect of Anakinra on Insulin Sensitivity in T...,UNKNOWN,None,16.0,0.358087,0.4790,Medium
431,NCT02590016,Glucose Control During Labour in Gestational D...,UNKNOWN,None,50.0,0.349293,0.4747,Medium


## 4. Benchmark enrollment rate from completed trials

For each completed trial, `enrollment_count / months(start -> primary completion)`
gives a rough monthly enrollment rate. The raw distribution is heavily
right-skewed (a handful of large registries enroll thousands per month), so we
winsorize at the 90th percentile before computing mean/std for the simulation —
a real data-cleaning step, not just cosmetic.

In [6]:
completed = trials[trials["overall_status"] == "COMPLETED"].copy()
completed["duration_months"] = (
    completed["primary_completion_date_dt"] - completed["start_date_dt"]
).dt.days / 30.44
completed = completed[(completed["duration_months"] >= 1) & (completed["enrollment_count"] > 0)]
completed["monthly_rate"] = completed["enrollment_count"] / completed["duration_months"]

rate_cap = completed["monthly_rate"].quantile(0.90)
clipped_rates = completed["monthly_rate"].clip(upper=rate_cap)
rate_mean, rate_std = clipped_rates.mean(), clipped_rates.std()

print(f"Benchmark from {len(completed)} completed trials with usable dates:")
print(f"  monthly enrollment rate: mean={rate_mean:.1f}, std={rate_std:.1f} (winsorized at p90={rate_cap:.1f})")

Benchmark from 106 completed trials with usable dates:
  monthly enrollment rate: mean=8.5, std=10.8 (winsorized at p90=32.9)


## 5. Scenario simulation for a recruiting trial

Pick the largest currently-RECRUITING trial and simulate months-to-complete
enrollment under the benchmark rate, assuming enrollment starts from zero.

In [7]:
recruiting = trials[trials["overall_status"] == "RECRUITING"].sort_values("enrollment_count", ascending=False)
target_trial = recruiting.iloc[0]

sim = simulate_enrollment_completion(
    target_enrollment=int(target_trial["enrollment_count"]),
    current_enrollment=0,
    monthly_rate_mean=rate_mean,
    monthly_rate_std=rate_std,
)
summary = summarise_simulation(sim, target_date_horizon_months=24)

print(f"Trial {target_trial['nct_id']}: '{target_trial['brief_title']}'")
print(f"Target enrollment: {int(target_trial['enrollment_count'])}")
summary

Trial NCT07135570: '"My Eyes, My Light": Amar Chokh, Amar Alo'
Target enrollment: 20000


{'n_simulations': 10000,
 'p_complete_within_horizon': 0.0,
 'median_months_to_complete': None,
 'p10_months': None,
 'p90_months': None}

## 6. Save processed outputs

Persist the scored trial watchlist and enrollment-scenario summary to
`data/processed/clinicaltrials/` for downstream use (e.g. the Streamlit
dashboard), alongside a metadata sidecar.

In [8]:
processed_dir = Path("../data/processed/clinicaltrials")
processed_dir.mkdir(parents=True, exist_ok=True)

trial_cols = ["nct_id", "brief_title", "overall_status", "why_stopped", "enrollment_count",
               "phase_primary", "study_type", "lead_sponsor_class", "termination_risk_model",
               "reporting_risk", "scale_risk", "risk_score", "risk_band"]
scored[trial_cols].to_csv(processed_dir / "trial_watchlist.csv", index=False)

simulation_summary = {
    "target_trial_nct_id": target_trial["nct_id"],
    "target_trial_title": target_trial["brief_title"],
    "target_enrollment": int(target_trial["enrollment_count"]),
    "benchmark_monthly_rate_mean": float(rate_mean),
    "benchmark_monthly_rate_std": float(rate_std),
    **summary,
}
with open(processed_dir / "enrollment_simulation.json", "w", encoding="utf-8") as f:
    json.dump(simulation_summary, f, indent=2)

with open(processed_dir / "processed.meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "therapeutic_area": THERAPEUTIC_AREA,
        "conditions_combined": CONDITIONS,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "files": {
            "trial_watchlist.csv": f"{len(scored)} trials, risk-scored",
            "enrollment_simulation.json": "Monte Carlo enrollment-completion scenario for the largest recruiting trial",
        },
    }, f, indent=2)

print(f"Wrote {len(scored)} trial rows and simulation summary to {processed_dir.resolve()}")

Wrote 478 trial rows and simulation summary to C:\Users\allif\Downloads\Risk Modelling\data\processed\clinicaltrials


## 7. Write stakeholder summary

In [9]:
from datetime import datetime, timezone
from pathlib import Path

top5 = scored.head(5)
n_critical = (scored["risk_band"] == "Critical").sum()
n_high = (scored["risk_band"] == "High").sum()
n_term = int(trials["target_terminated"].sum())
n_unknown = int(trials["reporting_risk"].sum())
area_desc = f"{THERAPEUTIC_AREA} ({len(CONDITIONS)} related conditions combined)"

lines = []
lines.append(f"# Clinical Trial Risk & Enrollment Scenario — {area_desc}\n")
lines.append(f"_Generated {datetime.now(timezone.utc).date()} from {len(trials)} unique trials "
              f"registered on ClinicalTrials.gov across {len(CONDITIONS)} related "
              f"'{THERAPEUTIC_AREA}' conditions._\n")
lines.append("## What this means\n")
lines.append(f"Of {len(trials)} trials reviewed, {n_term} have been terminated, withdrawn, or "
              f"suspended, and {n_unknown} have an 'Unknown' status (the sponsor has stopped "
              f"updating the registry). {n_critical} trial(s) are flagged **Critical** and "
              f"{n_high} **High** risk overall.\n")
lines.append("## Termination-risk classifier\n")
lines.append(f"A RandomForest classifier trained on {len(train_eligible)} trials with a settled "
              f"status ({n_term} terminated/withdrawn/suspended, {term_y.mean():.1%}) predicts "
              f"P(termination/withdrawal/suspension) from enrollment target, phase, study type, "
              f"sponsor class, and FDA-regulation flags. 5-fold cross-validated ROC-AUC: "
              f"**{term_cv_roc_auc:.2f}**. This is the dominant factor (weight 0.5) in the risk "
              f"score below. Model saved to "
              f"`models/clinical_trial_termination_classifier.joblib`.\n")
lines.append("## Highest-risk trials\n")
lines.append("| NCT ID | Title | Status | Why stopped | Termination risk (model) | Risk band |")
lines.append("|---|---|---|---|---|---|")
for _, r in top5.iterrows():
    title = (r["brief_title"] or "")[:60]
    why = r["why_stopped"] or "-"
    lines.append(f"| {r['nct_id']} | {title} | {r['overall_status']} | {why} | "
                  f"{r['termination_risk_model']:.0%} | {r['risk_band']} |")
lines.append("")
lines.append("## Enrollment scenario for the largest recruiting trial\n")
lines.append(f"**{target_trial['nct_id']}** ({target_trial['brief_title'][:80]}) is targeting "
              f"**{int(target_trial['enrollment_count'])}** participants. Using a benchmark "
              f"enrollment rate of **{rate_mean:.1f} participants/month** (derived from "
              f"completed {area_desc} trials, winsorized to remove outlier registries):\n")
if summary["median_months_to_complete"] is not None:
    lines.append(f"- Median time to complete enrollment: **{summary['median_months_to_complete']:.0f} months**\n")
    lines.append(f"- 80% confidence range: **{summary['p10_months']:.0f}–{summary['p90_months']:.0f} months**\n")
    lines.append(f"- Probability of completing enrollment within 24 months: "
                  f"**{summary.get('p_complete_by_target', 0.0):.0%}**\n")
else:
    lines.append(f"- At this benchmark rate, **0% of {summary['n_simulations']:,} simulations** "
                  f"completed enrollment within the 60-month simulation horizon — this trial's "
                  f"target enrollment is large relative to the typical recruitment pace observed "
                  f"in this therapeutic area. This itself is a risk signal worth escalating to "
                  f"the sponsor (re-baseline the target, add sites, or extend the timeline).\n")
lines.append("## Important caveats\n")
lines.append("- 'Enrollment count' on ClinicalTrials.gov is typically the *target*, not actual "
              "enrollment-to-date — this simulation assumes enrollment starts from zero.\n")
lines.append("- The benchmark rate is derived from a small, condition-specific sample and is "
              "sensitive to a handful of large registry-style studies — treat it as a planning "
              "reference, not a forecast.\n")
lines.append(f"- The termination-risk classifier was trained on only {n_term} positive cases — "
              "treat its scores as a triage aid, not a precise probability.\n")
lines.append("- 'Unknown' status often reflects administrative lapses, not necessarily trial "
              "failure — but it removes visibility for sponsors and regulators alike.\n")
lines.append("## Recommended next steps\n")
lines.append("1. Sponsor to follow up on 'Unknown'-status trials to confirm current status.\n")
lines.append("2. Review terminated/withdrawn trials' `why_stopped` text for recurring causes "
              "(funding, recruitment, safety) to inform protocol design for new trials.\n")
lines.append("3. Re-run the enrollment scenario quarterly with actual enrollment-to-date once "
              "available, to refine the forecast.\n")

report_path = Path("../reports/clinicaltrials_summary.md")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {report_path.resolve()}")

Wrote C:\Users\allif\Downloads\Risk Modelling\reports\clinicaltrials_summary.md
